# Dataset Creation - Synthetic CAPE Records

This notebook shows how the Syntheic dataset for the two-stage recommender was created. It separates the work into visible stages so that the configuration, curriculum inputs, graph construction, text processing, simulation and validation can be inspected independently

All records here are sythetic, they support controlled testing of the recommendation pipeline and are not estimates of real CAPE learner behaviour

Generated files are written to `generated_output/`

## 1. Import the required functions

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from generate_dataset import (
    build_tfidf_mapping,
    desired_resource_difficulty,
    generate_dataset,
    graph_features,
    load_inputs,
    prepare_output_directory,
    simulate_assessment,
    topological_order,
    validate_frames,
    write_outputs,
)

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config.json"
OUTPUT_DIR = PROJECT_ROOT / "generated_output"

required_paths = [
    CONFIG_PATH,
    PROJECT_ROOT / "seed_data/topics.csv",
    PROJECT_ROOT / "seed_data/prerequisites.csv",
    PROJECT_ROOT / "seed_data/resources.csv",
    PROJECT_ROOT / "seed_data/method_sources.csv",
]
missing = [str(path.name) for path in required_paths if not path.exists()]
assert not missing, f"Required input files are missing: {missing}"

print(f"Project folder: {PROJECT_ROOT}")
print("All required dataset-creation input files were found.")

Project folder: c:\Users\madma\Downloads\Complete 2 stage recommender system
All required dataset-creation input files were found.


## 2. Load the configuration and seed data

`config.json` has the random seed, sample size, interaction limits and the coefficients used by the simulator. 
Files in `seed_data/` has the ten selected topics, twelve provisional prerequisite links and forty external resources as CSVs

In [2]:
config, topics, prerequisites, resources, sources = load_inputs(CONFIG_PATH)

print("Scenario:", config["scenario"])
print("Random seed:", config["random_seed"])
print("Synthetic learners:", config["n_learners"])
print("Interactions per learner:", config["interaction_count"])
print("Diagnostic items per topic:", config["diagnostic"]["items_per_topic"])
print("Chronological holdout fraction:", config["chronological_holdout_fraction"])

Scenario: balanced_v1
Random seed: 3070
Synthetic learners: 500
Interactions per learner: {'minimum': 4, 'poisson_lambda': 4.0, 'maximum': 12}
Diagnostic items per topic: 8
Chronological holdout fraction: 0.2


In [3]:
input_summary = pd.DataFrame(
    {
        "input": ["topics", "prerequisites", "resources", "method sources"],
        "rows": [len(topics), len(prerequisites), len(resources), len(sources)],
        "columns": [len(topics.columns), len(prerequisites.columns), len(resources.columns), len(sources.columns)],
    }
)
print(input_summary.to_string(index=False))

resource_counts = resources.groupby("primary_topic_id").size()
assert len(topics) == 10
assert len(prerequisites) == 12
assert len(resources) == 40
assert resource_counts.eq(4).all()
print("\nEach topic has exactly four proposed external resources.")

         input  rows  columns
        topics    10        7
 prerequisites    12        4
     resources    40       17
method sources    13        8

Each topic has exactly four proposed external resources.


### Selected topic inputs

In [4]:
print(
    topics[["topic_id", "topic_name", "curriculum_order", "relative_difficulty"]]
    .sort_values("curriculum_order")
    .to_string(index=False)
)

topic_id      topic_name  curriculum_order  relative_difficulty
     T01       Variables                 1                    1
     T02       Selection                 2                    1
     T03       Iteration                 3                    2
     T04          Arrays                 4                    2
     T05       Functions                 5                    2
     T06      Algorithms                 6                    2
     T07 File Processing                 7                    3
     T08    Trace Tables                 8                    2
     T09   Boolean Logic                 9                    2
     T10     Logic Gates                10                    2


## 3. Check the prerequisite graph

Each prerequisite is stored as an edge from the topic that should be addressed first to the dependent topic. A topological ordering is calculated before graph features are used. If the graph contains a cycle, dataset generation stops with an error.

In [5]:
edges = list(
    prerequisites[["prerequisite_topic_id", "dependent_topic_id"]]
    .itertuples(index=False, name=None)
)
topic_order = topological_order(topics["topic_id"].tolist(), edges)
graph_table = graph_features(topics, prerequisites)

print("Topological order:")
print(" -> ".join(topic_order))
print("\nCalculated graph features:")
print(graph_table.round(6).to_string(index=False))

Topological order:
T01 -> T06 -> T09 -> T02 -> T03 -> T05 -> T10 -> T04 -> T08 -> T07

Calculated graph features:
topic_id  prerequisite_in_degree  dependent_out_degree  total_degree_centrality  downstream_topic_count  reversed_pagerank
     T01                       0                     4                 0.444444                       6           0.313454
     T02                       1                     1                 0.222222                       1           0.072843
     T03                       1                     3                 0.444444                       3           0.119884
     T04                       2                     1                 0.333333                       1           0.072843
     T05                       1                     1                 0.222222                       1           0.072843
     T06                       0                     1                 0.111111                       1           0.072843
     T07                 

The graph table records direct prerequisite and dependent counts, total degree centrality, the number of downstream topics and reversed PageRank. Reversed PageRank gives foundational topics credit from topics that depend on them.

## 4. Calculate TF-IDF resource-topic similarity

Topic text combines the topic name and syllabus-objective wording. 
Resource text combines the title, description, learning objective and programming-language alignment. Word unigrams and bigrams ar weighted with TF-IDF and compared using cosine similarity.

In [6]:
similarity_table, similarity_lookup = build_tfidf_mapping(topics, resources)

print("Resource-topic combinations:", len(similarity_table))
print("Minimum similarity:", round(similarity_table["tfidf_cosine_similarity"].min(), 6))
print("Maximum similarity:", round(similarity_table["tfidf_cosine_similarity"].max(), 6))

top_matches = (
    similarity_table.merge(resources[["resource_id", "title"]], on="resource_id")
    .merge(topics[["topic_id", "topic_name"]], on="topic_id")
    .sort_values("tfidf_cosine_similarity", ascending=False)
    .head(10)
)
print("\nTen highest lexical matches:")
print(
    top_matches[["title", "topic_name", "tfidf_cosine_similarity"]]
    .round({"tfidf_cosine_similarity": 4})
    .to_string(index=False)
)

Resource-topic combinations: 400
Minimum similarity: 0.0
Maximum similarity: 0.325696

Ten highest lexical matches:
                      title    topic_name  tfidf_cosine_similarity
Digital Logic Gate Tutorial   Logic Gates                   0.3257
             Operators in C Boolean Logic                   0.2998
                C Functions     Functions                   0.2403
                  Functions     Functions                   0.2263
     Logical Operators in C Boolean Logic                   0.2109
Algorithms and Trace Tables  Trace Tables                   0.1906
      C if...else Statement     Selection                   0.1899
         Using Trace Tables  Trace Tables                   0.1743
        Variables and Types     Variables                   0.1715
                C Operators Boolean Logic                   0.1711


TF–IDF similarity is treated as evidence of lexical alignment only. It must be noted that is does not establish that a resource is educationally effective

## 5. Demonstrate the diagnostic and difficulty rules

The baseline diagnostic contains eight simulated items for each learner-topic pair. Scores occur in increments of 12.5. Resource difficulty is divided into three operational bands based on the current topic score.

In [7]:
demonstration_rng = np.random.default_rng(config["random_seed"])
correct, item_count, score, _ = simulate_assessment(
    demonstration_rng,
    mastery=0.55,
    topic_relative_difficulty=2,
    diagnostic_config=config["diagnostic"],
)
print(f"Example diagnostic: {correct}/{item_count} correct = {score:.1f}")

example_scores = [25.0, 44.9, 45.0, 60.0, 74.9, 75.0, 87.5]

band_examples = pd.DataFrame(
    {
        "current_score": example_scores,
        "desired_resource_difficulty": [
            desired_resource_difficulty(value)
            for value in example_scores
        ],
    }
)
print("\nDifficulty-band examples:")
print(band_examples.to_string(index=False))

Example diagnostic: 4/8 correct = 50.0

Difficulty-band examples:
 current_score  desired_resource_difficulty
          25.0                            1
          44.9                            1
          45.0                            2
          60.0                            2
          74.9                            2
          75.0                            3
          87.5                            3


## 6. Generate the complete linked dataset

The next cell performs the main simulation. For each learner it creates the ten baseline topic records and then processes the learner's interactions in chronological order. Information from an interaction is added to history only after that interaction has been generated.

In [8]:
frames = generate_dataset(config, topics, prerequisites, resources)

generated_shapes = pd.DataFrame(
    [
        {"table": name, "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in frames.items()
    ]
).sort_values("table")

print(generated_shapes.to_string(index=False))
assert len(frames["learners"]) == 500
assert len(frames["learner_topic_baseline"]) == 5000
assert len(frames["interactions"]) == 3957
print("\nThe expected learner, baseline and interaction counts were produced.")

                       table  rows  columns
candidate_potential_outcomes 15828       12
                interactions  3957       18
      learner_topic_baseline  5000        6
         learner_topic_truth  5000        4
               learner_truth   500       12
                    learners   500        8
                 model_ready  2279       38
      model_ready_all_events  3957       38
               prerequisites    12        4
   resource_topic_similarity   400        5
             resource_topics    40        7
              resource_truth    40        4
                   resources    40       17
  selected_interaction_truth  3957       13
                      topics    10       12

The expected learner, baseline and interaction counts were produced.


### Inspect the generated learner and baseline records

In [9]:
print("Audit-profile counts:")
print(frames["learners"]["audit_profile_group"].value_counts().to_string())

print("\nFirst five learner-topic baseline records:")
print(frames["learner_topic_baseline"].head().to_string(index=False))

Audit-profile counts:
audit_profile_group
general                        450
high_mastery_low_confidence     13
low_mastery_high_confidence     13
persistent_support_need         12
capable_low_engagement          12

First five learner-topic baseline records:
learner_id topic_id  diagnostic_correct_items  diagnostic_item_count  diagnostic_score  confidence_score
     L0001      T01                         2                      8              25.0               2.6
     L0001      T02                         5                      8              62.5               2.0
     L0001      T03                         2                      8              25.0               1.5
     L0001      T04                         4                      8              50.0               2.7
     L0001      T05                         5                      8              62.5               2.8


### Inspect the time-ordered interactions

In [10]:
interactions = frames["interactions"]
interaction_view = [
    "interaction_id",
    "learner_id",
    "sequence_number",
    "simulated_day",
    "target_topic_id",
    "resource_id",
    "pre_score",
    "completed",
    "post_score",
    "score_gain",
    "chronological_split",
]

print(interactions[interaction_view].head(10).to_string(index=False))
print("\nChronological split counts:")
print(interactions["chronological_split"].value_counts().to_string())
print("\nObserved score-gain outcomes:", int(interactions["score_gain"].notna().sum()))
print("Missing score-gain outcomes:", int(interactions["score_gain"].isna().sum()))

interaction_id learner_id  sequence_number  simulated_day target_topic_id resource_id  pre_score  completed  post_score  score_gain chronological_split
       I000001      L0001                1              3             T07        R026        0.0          1         0.0         0.0       train_history
       I000002      L0001                2              7             T01        R001       25.0          1        37.5        12.5       train_history
       I000003      L0001                3             10             T03        R009       25.0          1        37.5        12.5       train_history
       I000004      L0001                4             15             T01        R003       37.5          1        50.0        12.5       train_history
       I000005      L0001                5             20             T07        R025        0.0          1        12.5        12.5       train_history
       I000006      L0001                6             26             T03        R011   

## 7. Check the public/internal separation

Generator-only tables contain latent traits and potential outcomes that are needed to audit the simulation. These hidden values are not allowed to become model predictors.

In [11]:
public_tables = [
    "topics",
    "prerequisites",
    "resources",
    "resource_topics",
    "resource_topic_similarity",
    "learners",
    "learner_topic_baseline",
    "interactions",
    "model_ready",
    "model_ready_all_events",
]
internal_tables = [
    "learner_truth",
    "learner_topic_truth",
    "resource_truth",
    "selected_interaction_truth",
    "candidate_potential_outcomes",
]
prohibited_tokens = ("latent", "oracle", "affinity", "quality", "appeal", "mastery")
model_columns = [column.lower() for column in frames["model_ready"].columns]
prohibited_predictors = sorted(
    column for column in model_columns
    if any(token in column for token in prohibited_tokens)
)

print("Public tables:", ", ".join(public_tables))
print("Generator-only tables:", ", ".join(internal_tables))
print("Prohibited generator-truth names in model_ready:", prohibited_predictors)
assert prohibited_predictors == []

Public tables: topics, prerequisites, resources, resource_topics, resource_topic_similarity, learners, learner_topic_baseline, interactions, model_ready, model_ready_all_events
Generator-only tables: learner_truth, learner_topic_truth, resource_truth, selected_interaction_truth, candidate_potential_outcomes
Prohibited generator-truth names in model_ready: []


## 8. Dataset validation checks

The validation stage checks the curriculum structure, identifiers, score ranges, outcome missingness, sparsity, chronological sequencing and the separation of hidden simulator values from model-ready fields.

In [12]:
validation_checks, validation_summary = validate_frames(frames, config)

print(validation_checks[["check", "status"]].to_string(index=False))
print("\nChecks passed:", validation_summary["check_count"] - validation_summary["failed_check_count"])
print("Checks failed:", validation_summary["failed_check_count"])
print("All checks passed:", validation_summary["all_checks_passed"])
assert validation_summary["all_checks_passed"]

                                     check status
             prerequisite_graph_is_acyclic   PASS
                               topic_count   PASS
                   prerequisite_edge_count   PASS
                            resource_count   PASS
                  four_resources_per_topic   PASS
                             learner_count   PASS
               learning_curve_subset_sizes   PASS
                         baseline_complete   PASS
                    unique_interaction_ids   PASS
                      foreign_key_learners   PASS
                     foreign_key_resources   PASS
           chronological_sequence_complete   PASS
                              score_ranges   PASS
                          confidence_range   PASS
                              rating_range   PASS
           post_outcome_missingness_policy   PASS
         no_generator_truth_in_model_ready   PASS
        no_current_outcome_as_feature_name   PASS
                 no_missing_model_features   PASS


## 9. Save dataset outputs

The output folder is cleaned before writing so that a rerun will not mix old and new generated files. Only `generated_output/` is removed; the notebook, configuration and seed inputs are not changed.

In [13]:
prepare_output_directory(OUTPUT_DIR, clean=True)
written_summary = write_outputs(frames, config, sources, OUTPUT_DIR)

saved_files = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
print("Output folder:", OUTPUT_DIR)
print("Files written:", len(saved_files))
print("All validation checks passed:", written_summary["all_checks_passed"])
print("\nMain saved tables:")
for path in sorted((OUTPUT_DIR / "data").glob("*.csv")):
    print(" -", path.relative_to(OUTPUT_DIR))

Output folder: c:\Users\madma\Downloads\Complete 2 stage recommender system\generated_output
Files written: 27
All validation checks passed: True

Main saved tables:
 - data\data_dictionary.csv
 - data\interactions.csv
 - data\learner_topic_baseline.csv
 - data\learners.csv
 - data\method_sources.csv
 - data\model_ready.csv
 - data\model_ready_all_events.csv
 - data\prerequisites.csv
 - data\resource_topic_similarity.csv
 - data\resource_topics.csv
 - data\resources.csv
 - data\topics.csv


## 10. Confirm same-seed reproducibility

The generator is executed a second time with the same configuration. Key tables from both runs must be identical. This checks the behaviour directly rather than assuming that setting a seed was sufficient.

In [14]:
repeat_frames = generate_dataset(config, topics, prerequisites, resources)
tables_checked = [
    "learners",
    "learner_topic_baseline",
    "interactions",
    "model_ready",
    "candidate_potential_outcomes",
]

for table_name in tables_checked:
    pd.testing.assert_frame_equal(frames[table_name], repeat_frames[table_name], check_exact=True)

print("Same-seed comparison passed for:")
for table_name in tables_checked:
    print(" -", table_name)

Same-seed comparison passed for:
 - learners
 - learner_topic_baseline
 - interactions
 - model_ready
 - candidate_potential_outcomes


## complete

500 synthetic learners, 5,000 learner-topic baseline records and 3,957 chronological interactions. and  the curriculum, resource, model-ready, validation and generator-only tables  weere created for the next stage.

